# PS6e6 TabM

Similar to XGBoost [notebook](https://www.kaggle.com/code/kirill0212/ps6e6-one-vs-rest-xgb), but with TabM model. This notebook results in 0.9686 CV and 0.9689 LB.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder, KBinsDiscretizer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score, recall_score, roc_curve
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
import gc
import os
from itertools import combinations
from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMRegressor,LGBMClassifier,log_evaluation,early_stopping
try:
    from cuml.ensemble import RandomForestClassifier
    from cuml.neighbors import KNeighborsClassifier
    from cuml.preprocessing import StandardScaler
    from cuml.pipeline import Pipeline
    import cuml
    cuml.set_global_output_type('numpy')
except:
    pass

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

ID = 'id'
TARGET = 'class'
NUMS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
CATS = ['spectral_type', 'galaxy_population']

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
SEED_LIST = [60, 0, 2809]
ADD_EXT = False
MODEL_TYPES_1 = [
                   # 'x',
                   # 'r',
                   't',
                   # 'l'
                  ]
MODEL_TYPES_2 = [
                   # 'x',
                   # 'r',
                   't',
                   # 'l',
                   # 'c', 
                   # 'h',
                   # 'knn',
                   # 'rf',  
                   # 'lr'
                  ]
MODEL_TYPES_3 = [
                   # 'x',
                   # 'r',
                   't',
                   # 'l'
                  ]

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

In [2]:
PATH = '/kaggle/input/competitions/playground-series-s6e6'
train = pd.read_csv(f'{PATH}/train.csv', index_col='id')
test = pd.read_csv(f'{PATH}/test.csv', index_col='id')
train[TARGET] = train[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})

In [3]:
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
num_cols = train.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in [ID, TARGET]]
num_cols = [col for col in num_cols if col not in [ID, TARGET]]
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
color_pairs = [
    ('u', 'g'),
    ('u', 'r'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_g_/_redshift'] = (df['g'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i'] / (df['redshift'] + 1e-6)).astype('float32')
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    bin_config = {'delta': [100, 500]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')
    return df

comb = feature_engineering(pd.concat([train.drop(columns=TARGET), test], ignore_index=True), fit=True)
y = train[TARGET].copy()
test = comb.iloc[len(y):].reset_index(drop=True)
train = comb.iloc[:len(y)].reset_index(drop=True)
train[TARGET] = y.copy()
del comb 
gc.collect()
new_cat_cols = [col for col in train.columns if col.endswith('_')]
new_num_cols = [col for col in train.columns if col.startswith('_')]

init len(cat_cols): 2
init len(num_cols): 8 



In [4]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)
new_cat_cols = [col for col in new_cat_cols if col not in DROP]

CATEGORY=CATS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+NUMS
train.head()

DROP:[]


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,_g_/_redshift,_i_/_redshift,_u-g,_u-r,alpha_cat_,delta_cat_,u_cat_,g_cat_,r_cat_,i_cat_,z_cat_,redshift_cat_,delta_100_quantile_bin_,delta_500_quantile_bin_,class
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0,0,53.536564,47.085331,3.576564,5.114196,0,0,0,0,0,0,0,0,43,215,0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0,0,120.821922,109.041748,1.691447,3.191300,1,1,1,1,1,1,1,0,66,331,0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,3,1,7.464886,7.289058,-0.043925,-0.136637,2,2,2,0,2,2,2,1,71,358,1
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0,0,39.266460,34.257919,2.254320,4.287302,3,3,3,0,3,3,3,0,91,457,0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0,0,35.035980,32.207016,2.231478,3.468709,4,4,2,1,4,1,3,0,46,233,0


In [5]:
TE_columns = []

columns = NUMS + CATS
good_columns = NUMS + CATS

important_combos = [
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
]
important_combos = sorted(important_combos)

for cols in important_combos:
    name = '-'.join(cols)

    train[name] = train[cols[0]].astype(str)
    for col in cols[1:]:
        train[name] = train[name] + '_' + train[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + '_' + test[col].astype(str)

    combined = pd.concat([train[name], test[name]], ignore_index=True)
    combined, _ = combined.factorize()
    if pd.Series(combined).nunique() > len(combined) // 2:
        train = train.drop(name, axis=1)
        test = test.drop(name, axis=1)
        continue
    train[name] = combined[:len(train)]
    test[name] = combined[len(train):len(train) + len(test)]

    TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

In [6]:
cat_cols = CATS + new_cat_cols

target_mapping = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
reverse_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
train_full = train.copy()
test_df = test.copy().reset_index(drop=True)
del train, test
gc.collect()
train_full['id'] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
submission = pd.read_csv(f'{PATH}/sample_submission.csv')
test_df['id'] = submission['id']

feature_cols = [c for c in train_full.columns if c not in ['id', TARGET] and train_full[c].dtype != 'object']
feature_cols = cat_cols + [c for c in feature_cols if c not in cat_cols]

print(f'Total features: {len(feature_cols)}')
print(f'Engineered features: {[c for c in feature_cols if c not in CATS + NUMS]}')

Total features: 26
Engineered features: ['alpha_cat_', 'delta_cat_', 'u_cat_', 'g_cat_', 'r_cat_', 'i_cat_', 'z_cat_', 'redshift_cat_', 'delta_100_quantile_bin_', 'delta_500_quantile_bin_', '_g_/_redshift', '_i_/_redshift', '_u-g', '_u-r', 'alpha_cat_-delta_cat_', 'u_cat_-z_cat_']


In [7]:
y = train_full[TARGET].map(target_mapping).values

for col in cat_cols:
    combined_cats = pd.concat([train_full[col], test_df[col]]).astype(str).unique()
    train_full[col] = pd.Categorical(train_full[col].astype(str), categories=combined_cats)
    test_df[col] = pd.Categorical(test_df[col].astype(str), categories=combined_cats)

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df['id'].values

for c in cat_cols:
    features[c] = features[c].cat.codes.astype('int32')
    test_features[c] = test_features[c].cat.codes.astype('int32')

In [8]:
y_0 = (y == 0).astype('int')
y_1 = (y == 1).astype('int')
y_2 = (y == 2).astype('int')
target = pd.DataFrame({'GALAXY': y_0, 'QSO': y_1, 'STAR': y_2})

In [9]:
def to_logits(df):
    df_col = df.columns
    epsilon = 1e-7
    preds_x = np.clip(df, epsilon, 1 - epsilon)
    logits = np.log(preds_x / (1 - preds_x))
    preds_x = pd.DataFrame(logits)
    preds_x.columns = df_col
    return preds_x

def balanced_accuracy():
    def xgb_balanced_accuracy(y_true, y_pred):
        n_classes = 2
        y_pred_labels = np.argmax(y_pred.reshape(-1, n_classes), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)

    xgb_balanced_accuracy.__name__ = 'bal_ACC'
    return xgb_balanced_accuracy

params_x = {
    'subsample': 0.8747340624732572,
    'colsample_bytree': 0.5,
    'max_depth': 4,
    'learning_rate': 0.02,
    'max_bin': 1024,
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42
}

params_c = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 5000,
    'learning_rate': 0.03,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': 42,
    'early_stopping_rounds': 1000,
    'task_type': 'GPU',
    'devices': '0'
}

params_l = {
    'random_state': 60,
    'feature_pre_filter': False,
    'verbose': -1,
    'n_estimators': 20000,
    'learning_rate': 0.05,
    'max_depth': 3,
    'min_child_samples': 63,
    'subsample': 0.812763123433567,
    'colsample_bytree': 0.8029300829885024, 
    'num_leaves': 247, 
    'reg_alpha': 0.07094285437903122,
    'reg_lambda': 0.033039097703242495,
    'max_bin': 32000,
}

In [10]:
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [11]:
import random
import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)

In [12]:
params_t = {
    'device': 'cuda',
    'val_metric_name': '1-auc_ovr',
    'random_state': 42,
    'verbosity': 0,
    'tabm_k': 32,
    'num_emb_type': 'pwl',
    'd_embedding': 12,
    'batch_size': 64,
    'lr': 3e-3,
    'weight_decay': 2e-2,
    'n_epochs': 3,
    'dropout': 0.1,
    'd_block': 512,
    'n_blocks': 3
}

In [13]:
params_r = {
    # --- Model architecture ---
    'val_metric_name':     '1-auc_ovr',
    "n_ens":                8,                # Number of ensemble members inside one RealMLP
    "embedding_size":       8,                # Embedding dim for high-cardinality categoricals
    "max_one_hot_cat_size": 8,               # Categoricals with nunique <= this get one-hot encoded
    "hidden_sizes":         [512, 512, 512],  # MLP hidden layer sizes
    "p_drop":               0.05,             # Base dropout probability value
    "p_drop_sched":         "expm4t",         # 'expm4t' | 'flat_cos' | 'constant'
    'act':                  'silu',          # Activation class for MLP layers
    "add_front_scale":      True,           # Whether to prepend a ScalingLayer before MLP layers

    # --- PBLD (periodic) embedding for numericals ---
    "plr_hidden_1": 20,        # plr_hidden_1 in official pytabkit's API
    "plr_hidden_2":    4,         # plr_hidden_2 + 1 (includes residual raw feature)
    "plr_sigma": 5.0,       # plr_sigma: init std of frequency weights
    "plr_act_name": 'prelu',  # plr_act_name: activation inside PBLD
    "plr_lr_factor":  0.093,     # plr_lr_factor: PBLD param lr = lr * pbld_lr_factor

    # --- Optimizer ---
    "lr":               0.01,       # Base learning rate (weights)
    "mom":              0.9,        # Adam beta1
    "sq_mom":           0.98,       # Adam beta2
    "lr_sched":        "flat_cos",  # 'flat_cos' | 'flat_anneal' | 'cos' | 'constant'
    # "flat_ratio":       0.3,        # Flat fraction for flat_cos / flat_anneal schedules
    "first_layer_lr_factor": 1.0,   # lr multiplier for first MLP layer weights
    # "first_layer_wd_factor": 0.1,   # wd multiplier for first MLP layer weights
    "scale_lr_factor":    10.0,       # Scale-layer lr = lr * lr_scale_mult
    "bias_lr_factor":     0.1,        # Bias        lr = lr * lr_bias_mult
    "wd":     0.013,      # Base weight decay (weights)
    # "wd_scale_mult":    0.1,        # Scale-layer wd = weight_decay * wd_scale_mult
    "bias_wd_factor":     0.5,        # Bias        wd = weight_decay * wd_bias_mult
    # "grad_clip":        1.0,        # Graient clipping value

    # --- Label smoothing ---
    "ls_eps":       0.04,   # Base label smoothing epsilon
    "ls_eps_sched": "cos",  # 'cos' | 'sqrt_cos' | 'constant'

    # --- Preprocessing ---
    # Supported tfms (applied in order to numerical features):
    # 'median_center', 'robust_scale', 'smooth_clip', 'l2_normalize'
    # categorical tfms ('one_hot', 'embedding') are handled separately by the model
    "tfms": ["median_center", "robust_scale"],

    # --- Training loop ---
    "n_epochs":    6,
    "batch_size":  256,
    # "eval_bs":   10240,
    "verbosity": 2,      # 0 = silent, 1 = fold summary only, 2 = per-epoch

    # --- Early stopping ---
    "use_early_stopping":                     False,
    "early_stopping_additive_patience":       10,     # stop if epoch > best * mult + add
    "early_stopping_multiplicative_patience": 1,

    # --- Device ---
    "device":       "cuda",
    "random_state": 42,
}

In [14]:
def load_preds(path, expected_rows=None):
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)

        # Case 1: broken flattened CSV with one column of length expected_rows * 3
        if df.shape[1] == 1:
            vals = df.iloc[:, 0].values

            if expected_rows is not None:
                assert len(vals) == expected_rows * 3, (
                    f'{path}: one-column CSV has length {len(vals)}, '
                    f'expected {expected_rows * 3}'
                )

            assert len(vals) % 3 == 0, (
                f'{path}: one-column CSV length {len(vals)} is not divisible by 3'
            )

            return pd.DataFrame(vals.reshape(-1, 3))

        # Case 2: normal CSV; use last 3 columns as probabilities
        return pd.DataFrame(df.iloc[:, -3:].values[:expected_rows])

    arr = np.load(path)

    if arr.ndim == 3:
        arr = arr.mean(axis=0)

    return pd.DataFrame(arr)

STACKING_FILES = [
    # ('xgb-0', '/kaggle/input/notebooks/cdeotte/xgb-v0-for-s6e6/oof_xgb_cv.csv', '/kaggle/input/notebooks/cdeotte/xgb-v0-for-s6e6/test_xgb_preds.csv'),
    # ('xgb-1', '/kaggle/input/notebooks/cdeotte/xgb-v1-for-s6e6/oof_preds.npy', '/kaggle/input/notebooks/cdeotte/xgb-v1-for-s6e6/test_preds.npy'),
    # ('realmlp-0', '/kaggle/input/notebooks/yekenot/ps-s6-e6-realmlp-pytorch/oof_preds.csv', '/kaggle/input/notebooks/yekenot/ps-s6-e6-realmlp-pytorch/test_preds.csv'),
    # ('realmlp-1', '/kaggle/input/notebooks/cdeotte/realmlp-v1-for-s6e6/oof_preds.npy', '/kaggle/input/notebooks/cdeotte/realmlp-v1-for-s6e6/test_preds.npy'),
    # ('tabm-0', '/kaggle/input/notebooks/donmarch14/s6e6-tabm/oof_preds.csv', '/kaggle/input/notebooks/donmarch14/s6e6-tabm/test_preds.csv'),
    # ('cat-0', '/kaggle/input/notebooks/cdeotte/cat-v0-for-s6e6/catboost_oof_predictions.csv', '/kaggle/input/notebooks/cdeotte/cat-v0-for-s6e6/catboost_test_predictions.csv'),
]

N = len(y)
M = len(test_ids)

add_oof = []
add_test = []
add_features = []
for name, oof_f, test_f in STACKING_FILES:
    try:
        o = load_preds(oof_f, expected_rows=N)

        t = load_preds(test_f, expected_rows=M)

        assert o.shape == (N, 3), f'OOF shape {o.shape} != {(N, 3)}'
        assert t.shape == (M, 3), f'Test shape {t.shape} != {(M, 3)}'

        add_oof.append(o)
        add_test.append(t)
        add_features.append(name)

        print(f'  {name:20s} OOF={o.shape} TEST={t.shape}')

    except Exception as e:
        print(f'  SKIP {name} ({oof_f}, {test_f}): {e}')

L = len(add_oof)

In [15]:
def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]
    
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-9:
                        best_bias, best_score, improved = c, s, True
                        history.append(best_score)
    return best_bias, best_score, history

In [16]:
oof_acc = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    best_bias, tuned_ba, opt_history = tune_bias(add_oof[i], y)
    oof_acc.loc[i] = [add_features[i], tuned_ba]
print(oof_acc.sort_values(by='score'))

0it [00:00, ?it/s]

Empty DataFrame
Columns: [model, score]
Index: []


In [17]:
oof_auc_0 = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_0 = roc_auc_score(y_0, add_oof[i].iloc[:, 0])
    oof_auc_0.loc[i] = [add_features[i], auc_0]
print(oof_auc_0.sort_values(by='score'))

0it [00:00, ?it/s]

Empty DataFrame
Columns: [model, score]
Index: []


In [18]:
oof_auc_1 = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_1 = roc_auc_score(y_1, add_oof[i].iloc[:, 1])
    oof_auc_1.loc[i] = [add_features[i], auc_1]
print(oof_auc_1.sort_values(by='score'))

0it [00:00, ?it/s]

Empty DataFrame
Columns: [model, score]
Index: []


In [19]:
oof_auc_2 = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_2 = roc_auc_score(y_2, add_oof[i].iloc[:, 2])
    oof_auc_2.loc[i] = [add_features[i], auc_2]
print(oof_auc_2.sort_values(by='score'))

0it [00:00, ?it/s]

Empty DataFrame
Columns: [model, score]
Index: []


In [20]:
s = {f'{i}': add_oof[i].iloc[:, 0] for i in range(L)}
pd.DataFrame(s).corr()

""


In [21]:
s = {f'{i}': add_test[i].iloc[:, 0] for i in range(L)}
pd.DataFrame(s).corr()

""


In [22]:
s = {f'{i}': add_oof[i].iloc[:, 1] for i in range(L)}
pd.DataFrame(s).corr()

""


In [23]:
s = {f'{i}': add_test[i].iloc[:, 1] for i in range(L)}
pd.DataFrame(s).corr()

""


In [24]:
s = {f'{i}': add_oof[i].iloc[:, 2] for i in range(L)}
pd.DataFrame(s).corr()

""


In [25]:
s = {f'{i}': add_test[i].iloc[:, 2] for i in range(L)}
pd.DataFrame(s).corr()

""


In [26]:
def run(model_type, target_id, drop_id, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target[target_name])):
        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_val_full = features.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
 
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_features.copy()
    
        encoder = TargetEncoder(cv=5, random_state=seed, smooth='auto')
        result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y[tr_idx][train_mask_2]))
        X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
        X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val_full[TE_columns]))
        X_val_full = pd.concat([result.reset_index(drop=True), X_val_full.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
        X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
        X_train = X_train.drop(TE_columns, axis=1)
        X_val = X_val.drop(TE_columns, axis=1)
        X_val_full = X_val_full.drop(TE_columns, axis=1)
        X_test = X_test.drop(TE_columns, axis=1)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        # print(X_train.shape)
        # print(X_val.shape)
        # print(X_val_full.shape)
        # print(X_test.shape)
        print(val_target.columns[i])
        
        if model_type == 't':
            model = TabM_D_Classifier(**params_t)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        elif model_type == 'r':
            model = RealMLP_TD_Classifier(**params_r)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i], cat_col_names=cat_cols)
        elif model_type == 'x':
            model = XGBClassifier(
                **params_x,
                n_estimators=20000,
                early_stopping_rounds=500,
            )

            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=[(X_val, val_target.iloc[:, i])],
                verbose=1000
            )
        elif model_type == 'c':
            model = CatBoostClassifier(**params_c)
    
            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=(X_val, val_target.iloc[:, i]),
                verbose=500
            )
        elif model_type == 'lr':
            model = LogisticRegressionCV(max_iter=10000, random_state=42, Cs=10, scoring='roc_auc', cv=5)

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'l':
            model=LGBMClassifier(**params_l)
    
            model.fit(X_train, y_train,eval_set=[(X_val, y_val)],
                      eval_metric='auc',
                      callbacks=[log_evaluation(500),early_stopping(250)])
        elif model_type == 'rf':
            model = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, max_features=0.3, output_type='numpy')

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'knn':
            X_train = X_train.astype('float32')
            X_val = X_val.astype('float32')
            X_test = X_test.astype('float32')
            model = Pipeline([
                ('scaler', StandardScaler()), 
                ('knn', KNeighborsClassifier(n_neighbors=200, weights='distance'))
            ])
            model.fit(X_train, train_target.iloc[:, i])
        elif model_type == 'h':
            model = HistGradientBoostingClassifier(**params_h)
            model.fit(X_train, train_target.iloc[:, i], X_val=X_val, y_val=val_target.iloc[:, i])

        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]

        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1_t = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1_t.to_parquet(f'oof_preds_{target_id}_{model_type}_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1_t = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1_t['id'] = test_ids
    test_preds_1_t.to_parquet(f'test_preds_{target_id}_{model_type}_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return h

In [27]:
def run_blend(target_id, drop_id, preds_tmp, test_tmp, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp, target[target_name]))):
        X_train = preds_tmp.iloc[tr_idx].reset_index(drop=True)
        X_val = preds_tmp.iloc[val_idx].reset_index(drop=True)
        X_val_full = preds_tmp.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
    
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_tmp.copy()
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
    
        model.fit(
            X_train, train_target.iloc[:, i]
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        map_coef = pd.Series(model.coef_[0], index=model.feature_names_in_)
        print(map_coef.sort_values())
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1 = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1.to_parquet(f'oof_preds_{target_id}_blend_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1 = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1['id'] = test_ids
    test_preds_1.to_parquet(f'test_preds_{target_id}_blend_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return oof_preds_1, test_preds_1

In [28]:
preds_tmp_1 = train_full[['id']]
preds_l_1 = []
test_l_1 = []
pred_col_1 = []
for model_type in MODEL_TYPES_1:
    for seed in SEED_LIST:
        run(model_type, 1, None, seed)
        preds_l_1.append(pd.read_parquet(f'oof_preds_1_{model_type}_{seed}.parquet')[['QSO']].rename(columns={'QSO': f'my_base_{model_type}_seed_{seed}'}))
        test_l_1.append(pd.read_parquet(f'test_preds_1_{model_type}_{seed}.parquet')[['QSO']].rename(columns={'QSO': f'my_base_{model_type}_seed_{seed}'}))
        pred_col_1.append(f'my_base_{model_type}_seed_{seed}')
if ADD_EXT:
    cnt_1 = 0
    for oof_tmp in add_oof:
        preds_l_1.append(oof_tmp.iloc[:, 1])
        pred_col_1.append(add_features[cnt_1])
        cnt_1 += 1
    for test_tmp in add_test:
        test_l_1.append(test_tmp.iloc[:, 1])
preds_tmp_1 = pd.concat(
                    preds_l_1
, axis=1)
test_tmp_1 = pd.concat(
                    test_l_1
, axis=1)
preds_tmp_1 = to_logits(preds_tmp_1)
test_tmp_1 = to_logits(test_tmp_1)
preds_tmp_1.columns = pred_col_1
test_tmp_1.columns = pred_col_1
print(preds_tmp_1.shape, test_tmp_1.shape, preds_tmp_1.mean(), test_tmp_1.mean(), preds_tmp_1.max(), test_tmp_1.max(), preds_tmp_1.min(), test_tmp_1.min())

QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9978205448007998
QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9978645265493649
QSO


...

0.9979174674731711
QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9980335764359543
QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.997927685815777
[np.float64(0.9979473957437874), np.float64(0.9978733517894388), np.float64(0.9979174674731711), np.float64(0.9980335764359543), np.float64(0.997927685815777)]
0.9979398954516258
(577347, 3) (247435, 3) my_base_t_seed_60     -4.731324
my_base_t_seed_0      -4.691157
my_base_t_seed_2809   -4.761645
dtype: float64 my_base_t_seed_60     -4.608121
my_base_t_seed_0      -4.599632
my_base_t_seed_2809   -4.660054
dtype: float64 my_base_t_seed_60      16.118096
my_base_t_seed_0       16.118096
my_base_t_seed_2809    15.942385
dtype: float64 my_base_t_seed_60      14.802950
my_base_t_seed_0       15.026094
my_base_t_seed_2809    14.237636
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64


In [29]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, preds_final_tmp = run_blend(1, None, preds_tmp_1, test_tmp_1, seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(preds_final_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_1 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_1.columns = target.columns
oof_preds_1['id'] = train_ids['id'].copy()
test_preds_1 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_1.columns = target.columns
test_preds_1['id'] = test_ids

0it [00:00, ?it/s]

QSO
0.9979260546707583
my_base_t_seed_2809    0.309166
my_base_t_seed_0       0.351189
my_base_t_seed_60      0.351535
dtype: float64


1it [00:08,  8.60s/it]

QSO
0.9979271295962213
my_base_t_seed_2809    0.322465
my_base_t_seed_60      0.340898
my_base_t_seed_0       0.356212
dtype: float64


2it [00:16,  8.10s/it]

...

QSO
0.9980390598100645
my_base_t_seed_2809    0.300012
my_base_t_seed_0       0.315994
my_base_t_seed_60      0.318404
dtype: float64


4it [00:32,  8.13s/it]

QSO
0.9979958084500224
my_base_t_seed_2809    0.320129
my_base_t_seed_60      0.333396
my_base_t_seed_0       0.354709
dtype: float64


5it [00:41,  8.22s/it]


[np.float64(0.9980085415200364), np.float64(0.9979283399307596), np.float64(0.9979569056583092), np.float64(0.9980390598100645), np.float64(0.9979958084500224)]
0.9979857310738384


In [30]:
preds_tmp_2 = train_full[['id']]
preds_l_2 = []
test_l_2 = []
pred_col_2 = []
for model_type in MODEL_TYPES_2:
    for seed in SEED_LIST:
        run(model_type, 2, None, seed)
        preds_l_2.append(pd.read_parquet(f'oof_preds_2_{model_type}_{seed}.parquet')[['STAR']].rename(columns={'STAR': f'my_base_{model_type}_seed_{seed}'}))
        test_l_2.append(pd.read_parquet(f'test_preds_2_{model_type}_{seed}.parquet')[['STAR']].rename(columns={'STAR': f'my_base_{model_type}_seed_{seed}'}))
        pred_col_2.append(f'my_base_{model_type}_seed_{seed}')
if ADD_EXT:
    cnt_2 = 0
    for oof_tmp in add_oof:
        preds_l_2.append(oof_tmp.iloc[:, 2])
        pred_col_2.append(add_features[cnt_2])
        cnt_2 += 1
    for test_tmp in add_test:
        test_l_2.append(test_tmp.iloc[:, 2])
preds_tmp_2 = pd.concat(
                    preds_l_2
, axis=1)
test_tmp_2 = pd.concat(
                    test_l_2
, axis=1)
preds_tmp_2 = to_logits(preds_tmp_2)
test_tmp_2 = to_logits(test_tmp_2)
preds_tmp_2.columns = pred_col_2
test_tmp_2.columns = pred_col_2
print(preds_tmp_2.shape, test_tmp_2.shape, preds_tmp_2.mean(), test_tmp_2.mean(), preds_tmp_2.max(), test_tmp_2.max(), preds_tmp_2.min(), test_tmp_2.min())

STAR


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9968550335569444
STAR


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9967681885191773
STAR


...

0.9967536037182106
STAR


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9969071131027152
STAR


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9968675945008728
[np.float64(0.9965616457302828), np.float64(0.9967723917577849), np.float64(0.9967536037182106), np.float64(0.9969071131027152), np.float64(0.9968675945008728)]
0.9967724697619731
(577347, 3) (247435, 3) my_base_t_seed_60     -8.841362
my_base_t_seed_0      -8.708259
my_base_t_seed_2809   -8.526074
dtype: float64 my_base_t_seed_60     -8.407473
my_base_t_seed_0      -8.447399
my_base_t_seed_2809   -8.032204
dtype: float64 my_base_t_seed_60      12.723506
my_base_t_seed_0       14.332947
my_base_t_seed_2809    16.118096
dtype: float64 my_base_t_seed_60      11.401742
my_base_t_seed_0       11.976815
my_base_t_seed_2809    11.519027
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64


In [31]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, test_preds_tmp = run_blend(2, None, preds_tmp_2, test_tmp_2, seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(test_preds_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_2 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_2.columns = target.columns
oof_preds_2['id'] = train_ids['id'].copy()
test_preds_2 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_2.columns = target.columns
test_preds_2['id'] = test_ids

0it [00:00, ?it/s]

STAR
0.9970000924871347
my_base_t_seed_60      0.320546
my_base_t_seed_2809    0.341955
my_base_t_seed_0       0.352791
dtype: float64


1it [00:07,  7.68s/it]

STAR
0.9969069045579355
my_base_t_seed_2809    0.307501
my_base_t_seed_60      0.308262
my_base_t_seed_0       0.317070
dtype: float64


2it [00:15,  7.68s/it]

...

STAR
0.9970152854994752
my_base_t_seed_2809    0.301786
my_base_t_seed_60      0.310997
my_base_t_seed_0       0.319055
dtype: float64


4it [00:32,  8.19s/it]

STAR
0.9969756165949729
my_base_t_seed_2809    0.302336
my_base_t_seed_60      0.313436
my_base_t_seed_0       0.318122
dtype: float64


5it [00:40,  8.09s/it]


[np.float64(0.9967108098367514), np.float64(0.9969488336883511), np.float64(0.9968768926257068), np.float64(0.9970152854994752), np.float64(0.9969756165949729)]
0.9969054876490515


In [32]:
preds_tmp_3 = train_full[['id']]
preds_l_3 = []
test_l_3 = []
pred_col_3 = []
for model_type in MODEL_TYPES_3:
    for seed in SEED_LIST:
        run(model_type, 0, None, seed)
        preds_l_3.append(pd.read_parquet(f'oof_preds_0_{model_type}_{seed}.parquet')[['GALAXY']].rename(columns={'GALAXY': f'my_base_{model_type}_seed_{seed}'}))
        test_l_3.append(pd.read_parquet(f'test_preds_0_{model_type}_{seed}.parquet')[['GALAXY']].rename(columns={'GALAXY': f'my_base_{model_type}_seed_{seed}'}))
        pred_col_3.append(f'my_base_{model_type}_seed_{seed}')
if ADD_EXT:
    cnt_3 = 0
    for oof_tmp in add_oof:
        preds_l_3.append(oof_tmp.iloc[:, 0])
        pred_col_3.append(add_features[cnt_3])
        cnt_3 += 1
    for test_tmp in add_test:
        test_l_3.append(test_tmp.iloc[:, 0])
preds_tmp_3 = pd.concat(
                    preds_l_3
, axis=1)
test_tmp_3 = pd.concat(
                    test_l_3
, axis=1)
preds_tmp_3 = to_logits(preds_tmp_3)
test_tmp_3 = to_logits(test_tmp_3)
preds_tmp_3.columns = pred_col_3
test_tmp_3.columns = pred_col_3
print(preds_tmp_3.shape, test_tmp_3.shape, preds_tmp_3.mean(), test_tmp_3.mean(), preds_tmp_3.max(), test_tmp_3.max(), preds_tmp_3.min(), test_tmp_3.min())

GALAXY


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9957817144100645
GALAXY


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9956199941069567
GALAXY


...

0.9956975568024153
GALAXY


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9957334024324638
GALAXY


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9958001445600659
[np.float64(0.9956840422750362), np.float64(0.9954076343991508), np.float64(0.9956975568024153), np.float64(0.9957334024324638), np.float64(0.9958001445600659)]
0.9956645560938264
(577347, 3) (247435, 3) my_base_t_seed_60      2.078496
my_base_t_seed_0       2.202460
my_base_t_seed_2809    2.151317
dtype: float64 my_base_t_seed_60      2.058834
my_base_t_seed_0       2.187747
my_base_t_seed_2809    2.133481
dtype: float64 my_base_t_seed_60      13.996474
my_base_t_seed_0       15.249238
my_base_t_seed_2809    14.843773
dtype: float64 my_base_t_seed_60      12.997944
my_base_t_seed_0       13.259651
my_base_t_seed_2809    13.591009
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64 my_base_t_seed_60     -16.118096
my_base_t_seed_0      -16.118096
my_base_t_seed_2809   -16.118096
dtype: float64


In [33]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, test_preds_tmp = run_blend(0, None, preds_tmp_3, test_tmp_3, seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(test_preds_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_3 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_3.columns = target.columns
oof_preds_3['id'] = train_ids['id'].copy()
test_preds_3 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_3.columns = target.columns
test_preds_3['id'] = test_ids

0it [00:00, ?it/s]

GALAXY
0.9958715081593329
my_base_t_seed_2809    0.290561
my_base_t_seed_60      0.345436
my_base_t_seed_0       0.393908
dtype: float64


1it [00:08,  8.63s/it]

GALAXY
0.9957174233560175
my_base_t_seed_2809    0.286747
my_base_t_seed_60      0.347580
my_base_t_seed_0       0.404313
dtype: float64


2it [00:17,  8.84s/it]

...

GALAXY
0.9958924002932665
my_base_t_seed_2809    0.291029
my_base_t_seed_60      0.346738
my_base_t_seed_0       0.393595
dtype: float64


4it [00:35,  8.82s/it]

GALAXY
0.9959513345848812
my_base_t_seed_2809    0.287332
my_base_t_seed_60      0.333823
my_base_t_seed_0       0.413048
dtype: float64


5it [00:43,  8.77s/it]


[np.float64(0.9958650943792706), np.float64(0.9955656908685039), np.float64(0.9958253282897715), np.float64(0.9958924002932665), np.float64(0.9959513345848812)]
0.9958199696831388


In [34]:
oof_final = oof_preds_1[['id', 'QSO']].join(oof_preds_2[['STAR']]).join(oof_preds_3[['GALAXY']])
test_final = test_preds_1[['id', 'QSO']].join(test_preds_2[['STAR']]).join(test_preds_3[['GALAXY']])

oof_final_2 = oof_final[['GALAXY', 'QSO', 'STAR']].copy()
test_final_2 = test_final[['GALAXY', 'QSO', 'STAR']].copy()
oof_final_2 = oof_final_2.div(oof_final_2.sum(axis=1), axis=0)
test_final_2 = test_final_2.div(test_final_2.sum(axis=1), axis=0)

oof_final_2.to_csv('oof_final.csv', index=False)
test_final_2.to_csv('test_final.csv', index=False)

In [35]:
def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]
    
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-9:
                        best_bias, best_score, improved = c, s, True
                        history.append(best_score)
    return best_bias, best_score, history
    
raw_oof_ba = balanced_accuracy_score(y, np.argmax(oof_final_2, axis=1))
print(f"📊 Raw OOF Balanced Accuracy: {raw_oof_ba:.6f}")

best_bias, tuned_ba, opt_history = tune_bias(oof_final_2, y)
print(f"✨ Found Optimal Biases [0, 1, 2]: {best_bias}")
print(f"📈 Tuned OOF Balanced Accuracy: {tuned_ba:.6f} (+{tuned_ba - raw_oof_ba:.6f})")

📊 Raw OOF Balanced Accuracy: 0.961011
✨ Found Optimal Biases [0, 1, 2]: [-1.42  -0.213  0.   ]
📈 Tuned OOF Balanced Accuracy: 0.968617 (+0.007606)


In [36]:
oof_next = public_preds(oof_final_2, best_bias)
oof_labels = [reverse_mapping[p] for p in oof_next]

test_next = public_preds(test_final_2, best_bias)
test_labels = [reverse_mapping[p] for p in test_next]

In [37]:
recalls = recall_score(y, oof_next, average=None)

print(f"Recall Galaxy:    {recalls[0]:.4f}")
print(f"Recall QSO: {recalls[1]:.4f}")
print(f"Recall Star:   {recalls[2]:.4f}")

bal_acc = recalls.mean()
print(f"Balanced Accuracy: {bal_acc:.5f}")

Recall Galaxy:    0.9574
Recall QSO: 0.9772
Recall Star:   0.9712
Balanced Accuracy: 0.96862


In [38]:
pd.Series(test_next).value_counts()

0    156332
1     51670
2     39433
Name: count, dtype: int64

In [39]:
cm = confusion_matrix(y, oof_next)
cm

array([[361416,   5772,  10292],
       [  1589, 114468,   1086],
       [  1999,    380,  80345]])

In [40]:
pd.Series(test_next).value_counts(normalize=True)

0    0.631810
1    0.208823
2    0.159367
Name: proportion, dtype: float64

In [41]:
pd.Series(oof_next).value_counts(normalize=True)

0    0.632209
1    0.208921
2    0.158870
Name: proportion, dtype: float64

In [42]:
pd.Series(y).value_counts(normalize=True)

0    0.653818
1    0.202899
2    0.143283
Name: proportion, dtype: float64

In [43]:
!rm *.parquet

In [44]:
import datetime

ts = datetime.datetime.now().strftime("_%Y%m%d_%H-%M-%S")
ts = ''

sub = pd.DataFrame({ID: test_ids, TARGET: test_labels})
sub.to_csv(f'submission{ts}.csv', index=False)
sub.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
